# 중간고사 대체 실습과제 — 문제 4

## 원인과 결과 찾기: 캠페인 인과효과 추정

| 항목 | 내용 |
|------|------|
| **관련 챕터** | Ch04. 원인과 결과 찾기 |
| **핵심 개념** | 상관 vs 인과, 교란변수, 인과효과(ATE) 추정 |
| **배점** | 25점 |
| **제출** | 이 노트북(.ipynb)을 실행 결과 포함하여 제출 |

---

## 시나리오

**프레시밀** 마케팅팀이 이메일 캠페인 결과를 보고합니다.

> *"캠페인 수신 고객의 구매율 42%, 미수신 고객은 22%.
> 캠페인 효과가 20%p입니다!"*

하지만 **구매성향이 높은 고객에게 캠페인을 더 많이 보냈기 때문에**
단순 비교는 과대추정일 수 있습니다. 진짜 효과를 추정하세요.

### 데이터 설명

| 열 | 설명 |
|-----|------|
| `customer_value` | 고객 구매성향 (0~1, 높을수록 잘 사는 고객) |
| `recency` | 마지막 구매 후 경과 일수 |
| `frequency` | 과거 구매 횟수 |
| `campaign` | 캠페인 수신 여부 (1=받음, 0=안받음) |
| `purchase` | 구매 여부 (1=구매, 0=미구매) |

### 인과 구조 (DAG)

```
  customer_value ──┐
                   ├──→ campaign ──→ purchase
  recency ─────────┤         ↗
                   ├────────┘
  frequency ───────┘
```

- `customer_value`, `recency`, `frequency`는 **교란변수**: 구매성향이 높은 고객에게 캠페인을 더 많이 보냄

In [1]:
# 환경설정
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

In [2]:
# 데이터 로드
df = pd.read_csv('../ch04/data/campaign.csv')
print(f'데이터 크기: {df.shape}')
print(f'캠페인 수신: {df["campaign"].sum()}명 / 미수신: {(df["campaign"]==0).sum()}명')
df.head()

데이터 크기: (5000, 5)
캠페인 수신: 2137명 / 미수신: 2863명


,customer_value,recency,frequency,campaign,purchase
0,0.3537,3.0,4,1,0
1,0.2486,10.6,5,0,0
2,0.4160,78.4,4,0,0
3,0.1600,16.8,5,0,0
4,0.5503,83.8,5,0,0


---

## 과제 1. 교란변수 확인 (8점)

### 요구사항

1. **(3점)** 캠페인 수신 그룹과 미수신 그룹의 `customer_value` 평균을 각각 계산하고,
   `customer_value` 분포를 **히스토그램**(두 그룹 겹쳐서 표시)으로 시각화하세요.

2. **(2점)** 두 그룹의 `customer_value` 평균 차이가 통계적으로 유의한지 **t-검정** 결과(p-value)를 `print()`로 출력하세요.

3. **(3점)** 단순 비교(42%-22%=20%p)가 **과대추정인 이유**를 교란변수 관점에서 2~3문장으로 설명하세요.

In [11]:
# ✏️ 과제 1-1: 그룹별 customer_value 평균 + 히스토그램
import plotly.express as px
import pandas as pd

# 1. 시각화를 위해 캠페인 수신 여부를 문자열로 매핑
df_plot = df.copy()
df_plot['campaign_group'] = df_plot['campaign'].map({1: '수신 그룹', 0: '미수신 그룹'})

# 2. 그룹별 customer_value 평균 계산
mean_values = df_plot.groupby('campaign_group')['customer_value'].mean()
print(f"--- customer_value 평균 ---")
print(f"미수신 그룹: {mean_values['미수신 그룹']:.4f}")
print(f"수신 그룹  : {mean_values['수신 그룹']:.4f}")

# 3. 히스토그램 시각화 (Plotly)
fig = px.histogram(
    df_plot, 
    x="customer_value", 
    color="campaign_group",
    barmode="overlay", # 두 그룹 겹쳐서 표시
    nbins=30,
    color_discrete_map={'수신 그룹': 'black', '미수신 그룹': 'gray'}, # 검정/회색 기조
    labels={'campaign_group': '그룹 구분', 'customer_value': '고객 구매 성향'}
)

# 4. 시각화 공통 규칙 적용
fig.update_layout(
    plot_bgcolor='white',         # 흰색 배경
    paper_bgcolor='white',        # 흰색 배경
    font=dict(color='black'),     # 모든 텍스트 검정
    title=dict(
        text="캠페인 수신 여부에 따른 customer_value 분포 비교", 
        x=0.5, 
        font=dict(size=18)
    ),
    xaxis=dict(
        showgrid=True, 
        gridcolor='lightgray', 
        linecolor='black', 
        tickfont=dict(color='black')
    ),
    yaxis=dict(
        showgrid=True, 
        gridcolor='lightgray', 
        linecolor='black',
        tickfont=dict(color='black')
    )
)

# 마커: 검정 테두리 설정 및 불투명도 조절 (겹침 확인용)
fig.update_traces(
    marker=dict(line=dict(width=1, color='black')), 
    opacity=0.6
)

fig.show()

--- customer_value 평균 ---
미수신 그룹: 0.2665
수신 그룹  : 0.3097


In [12]:
# ✏️ 과제 1-2: t-검정
from scipy import stats

# 1. 수신 그룹과 미수신 그룹의 데이터 분리
group_received = df[df['campaign'] == 1]['customer_value']
group_not_received = df[df['campaign'] == 0]['customer_value']

# 2. 독립표본 t-검정 수행 (Welch's t-test 적용을 위해 equal_var=False 권장)
t_stat, p_value = stats.ttest_ind(group_received, group_not_received, equal_var=False)

# 3. 결과 출력
print(f"--- t-test 결과 ---")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}") # 매우 작은 값일 수 있어 과학적 표기법 적용

# 4. 유의성 판단 결과 출력
alpha = 0.05
if p_value < alpha:
    print(f"\n결과: p-value가 {alpha}보다 작으므로, 두 그룹 간의 customer_value 평균 차이는 통계적으로 매우 유의합니다.")
    print("즉, 캠페인 수신 그룹이 미수신 그룹보다 원래 구매 성향이 높은 고객들로 구성되어 있습니다.")
else:
    print(f"\n결과: p-value가 {alpha}보다 크므로, 두 그룹 간의 customer_value 차이는 통계적으로 유의하지 않습니다.")

--- t-test 결과 ---
t-statistic: 9.6228
p-value: 1.0509e-21

결과: p-value가 0.05보다 작으므로, 두 그룹 간의 customer_value 평균 차이는 통계적으로 매우 유의합니다.
즉, 캠페인 수신 그룹이 미수신 그룹보다 원래 구매 성향이 높은 고객들로 구성되어 있습니다.


### ✏️ 과제 1-3: 과대추정 설명 (이 셀에 직접 작성)

(단순비교를 통한 20%p는 캠페인의 순수 효과뿐만이 아닌 원래 구매하려 했던 고객들의 특성이 포함된 결과다. customer_value 같은 교란변수가 캠페인의 수신 여부와 구매 여부 둘 다 영향을 주었다. 캠페인을 받지 않았어도 구매했을 고객들이 수신 그룹에 많이 포함되어 효과가 실제보다 크게 나타나는 집단 간 불균형이 발생했다.)

---

## 과제 2. 인과효과 추정 (10점)

교란변수를 통제하여 **진짜 캠페인 효과(ATE)**를 추정하세요.

### 요구사항

1. **(3점)** **단순 비교(Naive ATE)**: 캠페인 수신 vs 미수신 구매율 차이를 계산하세요.

2. **(7점)** **층화 분석으로 보정**: `customer_value`를 4분위(quartile)로 나누고,
   각 분위 안에서 캠페인 효과(수신 구매율 - 미수신 구매율)를 계산한 뒤,
   4개 분위의 **가중평균**으로 보정된 ATE를 구하세요.
   - 분위별 결과를 **표로 출력**
   - 보정된 ATE를 `print()`로 출력

In [14]:
# ✏️ 과제 2-1: 단순 비교 (Naive ATE)
naive_ate = df[df['campaign'] == 1]['purchase'].mean() - \
            df[df['campaign'] == 0]['purchase'].mean()

print(f"Naive ATE: {naive_ate:.4f}")

Naive ATE: 0.2009


In [15]:
# ✏️ 과제 2-2: 층화 분석 (customer_value 4분위)
# 각 분위별 캠페인 효과 → 가중평균 ATE
# 1. customer_value를 4분위수로 분할 (Q1, Q2, Q3, Q4)
df['strata'] = pd.qcut(df['customer_value'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# 2. 분위별 캠페인 효과 계산
# 각 분위(strata)와 캠페인 수신여부(campaign)에 따른 구매율(purchase) 평균 계산
stratified_summary = df.groupby(['strata', 'campaign'], observed=True)['purchase'].mean().unstack()
stratified_summary.columns = ['rate_0', 'rate_1']

# 분위별 효과(Effect) 및 샘플 크기(Weight) 계산
stratified_summary['effect'] = stratified_summary['rate_1'] - stratified_summary['rate_0']
stratified_summary['size'] = df.groupby('strata', observed=True).size()
stratified_summary['weight'] = stratified_summary['size'] / stratified_summary['size'].sum()

# 3. 보정된 ATE 계산 (분위별 효과의 가중평균)
adjusted_ate = (stratified_summary['effect'] * stratified_summary['weight']).sum()

# 결과 표 출력
print("--- [분위별 층화 분석 결과] ---")
print(stratified_summary[['rate_0', 'rate_1', 'effect', 'size']])
print(f"\n보정된 진짜 캠페인 효과(Adjusted ATE): {adjusted_ate:.4%}")

# 4. 분위별 효과 시각화 (Plotly)
fig = px.bar(
    stratified_summary.reset_index(), 
    x='strata', 
    y='effect',
    text_auto='.2%',
    labels={'strata': '고객 가치 분위(customer_value)', 'effect': '캠페인 효과 (구매율 차이)'},
    title="customer_value 분위별 보정된 캠페인 효과"
)

# 시각화 규칙 적용
fig.update_traces(
    marker_color='gray',                # 회색 기조
    marker_line_color='black',          # 검정 테두리
    marker_line_width=1.5,
    textfont=dict(color='black'),       # 텍스트 검정
    textposition='outside'
)

fig.add_hline(y=adjusted_ate, line_dash="dash", line_color="black", 
              annotation_text=f"가중평균 ATE: {adjusted_ate:.2%}", 
              annotation_position="top right")

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    xaxis=dict(showline=True, linecolor='black', gridcolor='lightgray'),
    yaxis=dict(showline=True, linecolor='black', gridcolor='lightgray', tickformat='.0%')
)

fig.show()

--- [분위별 층화 분석 결과] ---
          rate_0    rate_1    effect  size
strata                                    
Q1      0.141089  0.296380  0.155291  1250
Q2      0.203704  0.404040  0.200337  1251
Q3      0.222380  0.448529  0.226150  1250
Q4      0.349073  0.496951  0.147879  1249

보정된 진짜 캠페인 효과(Adjusted ATE): 18.2425%


### ✏️ 과제 2 해석 (이 셀에 직접 타이핑)

- 단순 비교 ATE와 층화 보정 ATE의 차이는 얼마인가?
1.76% 차이난다.
- 보정 후 수치가 참값(0.15)에 가까워졌는가?
보정 후 참값에 가까워졌지만 아직 차이가 존재한다.
- 이 차이가 발생한 이유를 교란변수 관점에서 1~2문장으로 설명하세요.
캠페인이 구매 성향이 높은 고객에 더 많이 할당되어 교란변수의 영향으로 단순 비교 결과보다 높은 값으로 평가되었다.

---

## 과제 3. 결과 해석 (7점)

### 요구사항

1. **(4점)** 두 가지 추정치를 비교하는 **막대 그래프**를 그리세요.
   - 막대: 단순 비교 ATE / 층화 보정 ATE / 참값(0.15)
   - 참값은 점선으로 표시해도 됩니다

2. **(3점)** 마케팅팀장에게 보내는 **분석 보고** 3문장을 작성하세요:
   - 단순 비교가 부정확한 이유
   - 보정된 진짜 효과 크기
   - 향후 캠페인 설계 시 주의할 점 1가지

In [19]:
# ✏️ 과제 3-1: 비교 막대 그래프
import plotly.graph_objects as go

# 1. 비교 데이터 설정
labels = ['단순 비교 ATE', '층화 보정 ATE']
values = [naive_ate, adjusted_ate] # 이전 단계에서 계산된 변수 활용
true_value = 0.15

# 2. 막대 그래프 생성
fig = go.Figure()

# ATE 막대 추가
fig.add_trace(go.Bar(
    x=labels,
    y=values,
    text=[f"{v:.2%}" for v in values],
    textposition='outside',
    marker_color=['lightgray', 'gray'], # 회색 기조
    marker_line_color='black',         # 검정 테두리
    marker_line_width=1.5,
    name='Estimated ATE'
))

# 3. 참값(0.15) 점선 추가
fig.add_hline(
    y=true_value, 
    line_dash="dash", 
    line_color="black", 
    line_width=2,
    annotation_text=f"참값 (Actual Effect: {true_value:.0%})", 
    annotation_position="top right",
    annotation_font=dict(color="black")
)

# 4. 시각화 공통 규칙 및 레이아웃 설정
fig.update_layout(
    title=dict(
        text="단순 비교 vs 층화 보정에 따른 캠페인 효과(ATE) 비교",
        x=0.5,
        font=dict(color='black', size=16)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    xaxis=dict(showline=True, linecolor='black'),
    yaxis=dict(
        showline=True, 
        linecolor='black', 
        gridcolor='lightgray',
        tickformat='.0%',
        range=[0, 0.25] # 비교를 위해 범위 고정
    ),
    showlegend=False
)

fig.show()


### ✏️ 과제 3-2: 마케팅팀장 보고 (이 셀에 직접 작성)

( 단순 비교가 부정확한 이유: 단순 비교는 구매 성향이 높은 고객에게 캠페인이 집중된 집단 간 불균형으로 인해 실제 효과보다 크게 나옴.
  보정된 진짜 효과 크기: 캠페인의 진짜 효과는 18.24% 로 나왔다.
  향후 캠페인 설계 시 주의할 점 1가지: 전체의 고객을 대상으로 하는 것보단 무작위 할당을 적용하는 등 편향을 최소화해야한다.)

---

## 채점 기준

| 과제 | 배점 | 채점 포인트 |
|------|------|------------|
| 과제 1. 교란변수 확인 | 8점 | 히스토그램(3) + t검정(2) + 설명(3) |
| 과제 2. 인과효과 추정 | 10점 | 단순비교(3) + 층화분석+보정ATE(7) |
| 과제 3. 결과 해석 | 7점 | 비교 그래프(4) + 보고서(3) |
| **합계** | **25점** | |

---

## 참고: 예상 정답값

- **진짜 캠페인 효과(True ATE)**: 0.15 (구매 확률 15%p 증가)
- **단순 비교**: ~0.20 (교란변수 때문에 과대추정)
- **층화 보정**: ~0.18 (참값 방향으로 보정됨)
- 캠페인 수신 그룹의 `customer_value` 평균이 미수신 그룹보다 유의하게 높음 (p < 0.05)